# Exploring Chant Sources with PyCantus

**Workshop Notebook 1 of 2**

This notebook introduces the **PyCantus** library for working with medieval chant data from the [Cantus Database](https://cantusdatabase.org/). We will:

1. Load a chant corpus from CSV files
2. Inspect sources (manuscripts)
3. Collect chants belonging to a specific source
4. Explore genres and feast names
5. Build summary DataFrames with Pandas

---

### Background: The Cantus Data Model

| Term | Meaning |
|---|---|
| **Source** | A physical manuscript or printed book containing chants |
| **Chant** | One occurrence of a chant in a source, with text and (optionally) a melody |
| **Corpus** | A collection of sources and chants, loaded from CSV |
| **Volpiano** | A text-based encoding of a chant melody |
| **Genre** | The liturgical type of a chant (A = Antiphon, R = Responsory, etc.) |

---

## 1. Installation & Imports

In [ ]:
# Run this cell once to install PyCantus
# (Skip if already installed)
!pip install git+https://github.com/dact-chant/PyCantus.git -q

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pycantus.models import Corpus

# Quick sanity check
from pycantus import hello_pycantus
hello_pycantus()

---
## 2. Loading a Corpus

A **Corpus** is loaded from two CSV files:
- `chants.csv` — one row per chant occurrence
- `sources.csv` — one row per manuscript source

The [CantusCorpus v1.0](https://github.com/DvorakovaA/CantusCorpus) dataset is the recommended starting point.  
Download it and update the file paths below, **or** let PyCantus download it automatically via the fallback URLs.

> **Tip:** For a workshop, you can use the small sample files included in the PyCantus repository under `tutorials/`.

In [ ]:
# -------------------------------------------------------------------
# Option A: Point to local CSV files (fastest)
# -------------------------------------------------------------------
# CHANTS_FILE = 'path/to/chants.csv'
# SOURCES_FILE = 'path/to/sources.csv'

# -------------------------------------------------------------------
# Option B: Let PyCantus download from the CantusCorpus GitHub release
# (requires internet access; may take a moment)
# -------------------------------------------------------------------
CHANTS_FILE  = 'chants.csv'
SOURCES_FILE = 'sources.csv'

CHANTS_URL  = 'https://raw.githubusercontent.com/DvorakovaA/CantusCorpus/main/data/chants.csv'
SOURCES_URL = 'https://raw.githubusercontent.com/DvorakovaA/CantusCorpus/main/data/sources.csv'

corpus = Corpus(
    chants_filepath=CHANTS_FILE,
    sources_filepath=SOURCES_FILE,
    chants_fallback_url=CHANTS_URL,
    sources_fallback_url=SOURCES_URL,
    create_missing_sources=True,   # tolerate chants whose source isn't listed
)

print(f"Loaded {len(corpus.chants):,} chants from {len(corpus.sources):,} sources.")

---
## 3. Inspecting Sources

Each `Source` object has these key attributes:

| Attribute | Description |
|---|---|
| `siglum` | Short manuscript identifier (e.g. `"A-Wn 1799"`) |
| `title` | Full title of the manuscript |
| `century` | Century of origin (e.g. `"12"`) |
| `provenance` | Place of origin |
| `cursus` | Secular or Monastic |
| `srclink` | URL in the Cantus Database |

In [ ]:
# Look at the first source
first_source = corpus.sources[0]
print(first_source)

In [ ]:
# Build a DataFrame of all sources for easy browsing
sources_df = pd.DataFrame([
    {
        'siglum':    s.siglum,
        'title':     s.title,
        'century':   s.century,
        'provenance': s.provenance,
        'cursus':    s.cursus,
        'srclink':   s.srclink,
    }
    for s in corpus.sources
])

print(f"Shape: {sources_df.shape}")
sources_df.head(10)

In [ ]:
# How many sources per century?
century_counts = (
    sources_df['century']
    .dropna()
    .astype(str)
    .value_counts()
    .sort_index()
)
century_counts

---
## 4. Selecting a Single Source

Pick a source by its **siglum** (the short manuscript code).  
Change `TARGET_SIGLUM` to any siglum you see in the table above.

In [ ]:
# Choose a source to explore
TARGET_SIGLUM = sources_df['siglum'].iloc[0]   # <-- change this!
print(f"Exploring source: {TARGET_SIGLUM}")

In [ ]:
# Collect all chants from that source into a list
source_chants = [
    c for c in corpus.chants
    if c.siglum == TARGET_SIGLUM
]

print(f"Found {len(source_chants)} chants in '{TARGET_SIGLUM}'")

In [ ]:
# Print a few chants to see their structure
for chant in source_chants[:3]:
    print(chant)
    print('-' * 40)

---
## 5. Building a Chant DataFrame

We turn the list of `Chant` objects into a Pandas DataFrame.  
This makes it easy to sort, filter, and visualise the data.

In [ ]:
# Key chant fields
CHANT_FIELDS = [
    'cantus_id', 'incipit', 'feast', 'genre',
    'office', 'position', 'mode', 'folio',
    'sequence', 'melody', 'chantlink'
]

def chant_to_dict(c):
    """Convert a Chant object to a plain dictionary."""
    return {field: getattr(c, field, None) for field in CHANT_FIELDS}

chants_df = pd.DataFrame([chant_to_dict(c) for c in source_chants])
print(f"DataFrame shape: {chants_df.shape}")
chants_df.head()

In [ ]:
# Quick summary of missing values
chants_df.isnull().sum()

---
## 6. Exploring Genres

Cantus uses single-letter genre codes. Common ones:

| Code | Genre |
|---|---|
| A | Antiphon |
| R | Responsory |
| V | Verse (of a Responsory) |
| H | Hymn |
| W | Versicle |
| I | Invitatory |
| G | Gradual |
| Of | Offertory |
| Co | Communion |

In [ ]:
# Map genre codes to readable names
GENRE_NAMES = {
    'A': 'Antiphon',
    'R': 'Responsory',
    'V': 'Verse',
    'H': 'Hymn',
    'W': 'Versicle',
    'I': 'Invitatory',
    'G': 'Gradual',
    'Of': 'Offertory',
    'Co': 'Communion',
    'Gr': 'Gradual Verse',
    'Tr': 'Tract',
    'Al': 'Alleluia',
}

chants_df['genre_name'] = chants_df['genre'].map(GENRE_NAMES).fillna(chants_df['genre'])

# Count by genre
genre_counts = chants_df['genre_name'].value_counts()
print(genre_counts)

In [ ]:
# Bar chart of genres
fig, ax = plt.subplots(figsize=(8, 4))
genre_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title(f'Chant Genres in {TARGET_SIGLUM}', fontsize=13)
ax.set_xlabel('Genre')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

---
## 7. Exploring Feasts and Chant Titles

The `incipit` is the opening phrase of the chant text — effectively its title.  
The `feast` field gives the liturgical occasion.

In [ ]:
# Most common feast names
feast_counts = chants_df['feast'].value_counts().head(15)
print("Top 15 feasts:")
print(feast_counts)

In [ ]:
# Show chant titles (incipits) for a specific feast
# Change this to any feast name you see above
SELECTED_FEAST = feast_counts.index[0]

feast_chants = chants_df[chants_df['feast'] == SELECTED_FEAST]

print(f"Chants for feast: '{SELECTED_FEAST}'  ({len(feast_chants)} total)\n")
feast_chants[['genre_name', 'incipit', 'mode', 'folio']].to_string(index=False)

In [ ]:
# Cross-tabulation: genres × top feasts
top_feasts = feast_counts.index[:8].tolist()
cross_tab = (
    chants_df[chants_df['feast'].isin(top_feasts)]
    .groupby(['feast', 'genre_name'])
    .size()
    .unstack(fill_value=0)
)
cross_tab

---
## 8. Modal Distribution

Medieval chant is classified into 8 **modes** (musical scales).  
Modes 1–4 are *authentic*; modes 5–8 are *plagal*.

In [ ]:
mode_counts = (
    chants_df['mode']
    .dropna()
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(7, 4))
mode_counts.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title(f'Modal Distribution in {TARGET_SIGLUM}', fontsize=13)
ax.set_xlabel('Mode')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

---
## 9. Chants with Melodies

Not all chants in the database have a melody encoded in Volpiano.  
Let's see how many do in our source.

In [ ]:
has_melody = chants_df['melody'].notna()
print(f"Chants with melody:    {has_melody.sum()} ({has_melody.mean()*100:.1f}%)")
print(f"Chants without melody: {(~has_melody).sum()}")

In [ ]:
# Preview the Volpiano strings for the first few chants that have one
melody_sample = chants_df[has_melody][['incipit', 'genre_name', 'mode', 'melody']].head(5)
melody_sample

---
## 10. Exporting Results

Save the DataFrame to CSV for further analysis.

In [ ]:
output_filename = f"chants_{TARGET_SIGLUM.replace(' ', '_').replace('/', '-')}.csv"
chants_df.to_csv(output_filename, index=False)
print(f"Saved to: {output_filename}")

---
## Summary

In this notebook you learned how to:

- Load a PyCantus `Corpus` from CSV files
- Browse `Source` metadata
- Filter chants by source siglum
- Build a Pandas DataFrame from `Chant` objects
- Explore genres, feast names, and modes with counts and charts
- Identify chants that have Volpiano melodies

➡️ Continue to **Notebook 2** to learn how to decode Volpiano melodies into tabular data.